# L09 – Spara, logga och felsöka

**Tid:** ca 2 h · **Del B: Träna ett nätverk väl**

## Mål
1. **Spara och ladda** modeller med `state_dict`, och spara checkpoints så att träningen kan återupptas.
2. **Logga experiment** med TensorBoard så att körningar kan jämföras.
3. Använda en **felsökningschecklista** när träningen inte fungerar.
4. Förstå kort **varför initieringen spelar roll**.

**Var i arbetsflödet?** Steg 10 (spara), men också genom hela flödet. Utan loggning kan du inte jämföra experiment, och utan felsökningsrutiner hittar du inte de *tysta* buggarna. Koden kör utan fel, men modellen blir dålig.

> Kräver `dlkit` från L08. I den här lektionen får `dlkit.fit` stöd för checkpoints och TensorBoard.

In [ ]:
import sys
sys.path.insert(0, "..")

import math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

from dlkit.seed import set_seed
from dlkit.data import fashion_mnist_loaders

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
loss_fn = nn.CrossEntropyLoss()
CKPT_DIR = Path("../checkpoints"); CKPT_DIR.mkdir(exist_ok=True)
RUNS_DIR = Path("../runs"); RUNS_DIR.mkdir(exist_ok=True)

def make_mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                         nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))

train_loader, val_loader, test_loader = fashion_mnist_loaders()

## 1. `state_dict` – modellens vikter

`model.state_dict()` är en ordbok från lagernamn till vikttensorer. Den innehåller *bara siffrorna*, inte koden för arkitekturen.

In [ ]:
from dlkit.train import fit, evaluate, predict

set_seed(42)
model = make_mlp().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
fit(model, train_loader, val_loader, loss_fn, optimizer, device, epochs=3)

for name, tensor in model.state_dict().items():
    print(f"{name:10s} {tuple(tensor.shape)}")

### Spara och ladda

**Rekommenderat sätt:** spara `state_dict`. För att ladda skapar du först *samma arkitektur* och fyller den sedan med vikterna.

**Varför inte `torch.save(model)`?** Det sparar hela objektet med *pickle*. Filen blir beroende av exakt den kodstruktur och de klassnamn du hade när du sparade, och den går lätt sönder när koden ändras. Att ladda pickle-filer från okända källor är dessutom en säkerhetsrisk, eftersom de kan köra godtycklig kod.

In [ ]:
path = CKPT_DIR / "mlp_l09.pt"
torch.save(model.state_dict(), path)
print(f"sparad: {path} ({path.stat().st_size / 1e6:.1f} MB)")

loaded = make_mlp().to(device)                           # 1. samma arkitektur
loaded.load_state_dict(torch.load(path, map_location=device, weights_only=True))   # 2. fyll med vikter
loaded.eval()                                            # 3. eval-läge innan prediktion!

_, pred_original, _ = predict(model, val_loader, device)
_, pred_loaded, _ = predict(loaded, val_loader, device)
print("identiska prediktioner:", (pred_original == pred_loaded).all())

- `map_location=device` gör att en modell som sparats på GPU kan laddas på en dator utan GPU.
- `weights_only=True` laddar bara tensorer och ingen godtycklig kod, vilket är det säkra sättet.

### Checkpoints för att återuppta träningen

För att kunna *fortsätta* träna, till exempel efter en krasch eller för att träna fler epoker, behövs mer än vikterna:
- **optimerarens tillstånd** (Adam minns tidigare gradienter),
- **epoken** du var på,
- gärna **historiken** och schedulerns tillstånd.

In [ ]:
ckpt_path = CKPT_DIR / "resume_demo.pt"
torch.save({
    "epoch": 3,
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
}, ckpt_path)

# ... senare, i en ny session:
ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
model2 = make_mlp().to(device)
model2.load_state_dict(ckpt["model"])
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3)
optimizer2.load_state_dict(ckpt["optimizer"])
start_epoch = ckpt["epoch"] + 1
print(f"fortsätter från epok {start_epoch}, val acc just nu: {evaluate(model2, val_loader, loss_fn, device)[1]:.3f}")
history2 = fit(model2, train_loader, val_loader, loss_fn, optimizer2, device, epochs=2)

## 2. Bygg in checkpoints och loggning i `dlkit`

Nu lägger vi till två saker i `fit`. Det nya är markerat med `# NEW (L09)`.
- `checkpoint_path`: spara den bästa modellen (och optimeraren) till disk varje gång val loss förbättras.
- `writer`: logga loss, accuracy och learning rate till TensorBoard.

In [ ]:
%%writefile ../dlkit/train.py
import copy
import time

import torch


def train_one_epoch(model, loader, loss_fn, optimizer, device):
    """One pass over the training data. Returns (mean loss, accuracy)."""
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    """Evaluate without gradients. Returns (mean loss, accuracy)."""
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += loss_fn(logits, y).item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def predict(model, loader, device):
    """Return (y_true, y_pred, probs) as NumPy arrays – ready for sklearn.metrics."""
    model.eval()
    ys, probs = [], []
    for X, y in loader:
        logits = model(X.to(device))
        probs.append(torch.softmax(logits, dim=1).cpu())
        ys.append(y)
    probs = torch.cat(probs)
    return torch.cat(ys).numpy(), probs.argmax(dim=1).numpy(), probs.numpy()


def fit(model, train_loader, val_loader, loss_fn, optimizer, device, epochs=10,
        patience=None, scheduler=None, checkpoint_path=None, writer=None, verbose=True):
    """Train for up to `epochs` epochs. Returns a history dict.

    patience:        early stopping – stop after this many epochs without val-loss
                     improvement and restore the best weights (L07).
    scheduler:       a torch.optim.lr_scheduler, stepped once per epoch (L08).
    checkpoint_path: save the best model + optimizer here whenever val loss improves.  # NEW (L09)
    writer:          a torch.utils.tensorboard.SummaryWriter for logging.              # NEW (L09)
    """
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    best_loss, best_state, best_epoch, bad_epochs = float("inf"), None, 0, 0
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        lr = optimizer.param_groups[0]["lr"]
        tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, loss_fn, device)
        for key, value in zip(history, (tr_loss, tr_acc, va_loss, va_acc, lr)):
            history[key].append(value)

        if writer is not None:  # NEW (L09)
            writer.add_scalar("loss/train", tr_loss, epoch)
            writer.add_scalar("loss/val", va_loss, epoch)
            writer.add_scalar("acc/train", tr_acc, epoch)
            writer.add_scalar("acc/val", va_acc, epoch)
            writer.add_scalar("lr", lr, epoch)

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(va_loss)
            else:
                scheduler.step()

        improved = va_loss < best_loss
        if improved:
            best_loss, best_epoch, bad_epochs = va_loss, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
            if checkpoint_path is not None:  # NEW (L09)
                torch.save({"epoch": epoch, "val_loss": va_loss, "model": best_state,
                            "optimizer": optimizer.state_dict()}, checkpoint_path)
        else:
            bad_epochs += 1

        if verbose:
            print(f"epok {epoch:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.3f} | lr {lr:.1e} | {time.time() - t0:.1f}s"
                  + ("  *" if improved else ""))

        if patience is not None and bad_epochs >= patience:
            if verbose:
                print(f"Early stopping: ingen förbättring på {patience} epoker.")
            break

    if patience is not None and best_state is not None:
        model.load_state_dict(best_state)
        if verbose:
            print(f"Återställde vikterna från epok {best_epoch} (val loss {best_loss:.4f}).")
    if writer is not None:  # NEW (L09)
        writer.flush()
    history["best_epoch"] = best_epoch
    return history

In [ ]:
import importlib, dlkit.train
importlib.reload(dlkit.train)
from dlkit.train import fit, evaluate, predict

## 3. Experimentloggning med TensorBoard

När du kör många experiment glömmer du snabbt vilken körning som var vilken. **Logga varje körning** under ett beskrivande namn, och spara konfigurationen bredvid resultatet.

Vi jämför två learning rates. Varje körning får en egen mapp under `runs/`:

In [ ]:
import json
from torch.utils.tensorboard import SummaryWriter

for lr in [1e-3, 1e-2]:
    config = {"model": "mlp_256_128", "optimizer": "Adam", "lr": lr, "batch_size": 128, "epochs": 8}
    run_name = f"mlp_adam_lr{lr:g}"
    writer = SummaryWriter(log_dir=str(RUNS_DIR / run_name))
    set_seed(42)
    model = make_mlp().to(device)
    h = fit(model, train_loader, val_loader, loss_fn, torch.optim.Adam(model.parameters(), lr=lr),
            device, epochs=config["epochs"], checkpoint_path=CKPT_DIR / f"{run_name}.pt",
            writer=writer, verbose=False)
    writer.close()
    # Spara konfigurationen och slutresultatet bredvid loggen, så vet du senare vad körningen var
    (RUNS_DIR / run_name / "config.json").write_text(json.dumps(
        {**config, "best_val_loss": min(h["val_loss"]), "best_val_acc": max(h["val_acc"])}, indent=2))
    print(f"{run_name}: bästa val acc {max(h['val_acc']):.3f}")

**Så öppnar du TensorBoard:**
- **I VS Code:** `Ctrl+Shift+P` och sedan *Python: Launch TensorBoard*. Välj mappen `dl-course/runs`.
- **I en terminal** (från repots rot, med `.venv` aktiverad): `tensorboard --logdir dl-course/runs`. Öppna sedan http://localhost:6006.

Under *Scalars* kan du lägga körningarna ovanpå varandra och jämföra dem. Det är samma inlärningskurvor som med `plot_history`, men för alla experiment på en gång.

**Ett alternativ utan verktyg:** spara `history` som JSON eller CSV per körning och plotta med pandas. Huvudsaken är att *alla* körningar loggas, inte vilket verktyg du använder. I större projekt används ofta MLflow eller Weights & Biases.

## 4. Felsökningschecklista: när träningen inte fungerar

Buggar i djupinlärning ger sällan felmeddelanden. Modellen blir bara sämre än den borde. Gå igenom stegen i ordning:

| # | Kontroll | Förväntat |
|---|---|---|
| 1 | **Titta på datan**: bilder, etiketter, shapes, dtype, värdeintervall efter normalisering | bilderna ser rätt ut och etiketterna matchar |
| 2 | **Loss vid start** med en otränad modell | ≈ $\ln K$ för $K$ klasser |
| 3 | **Överanpassa en enda batch** | loss går mot ≈ 0 |
| 4 | **Gradientnormer** de första stegen | rimliga tal, inte 0, inte `nan`, inte enorma |
| 5 | **Loss ökar eller blir `nan`** | sänk learning rate (L03) |
| 6 | **`model.train()` och `model.eval()`** på rätt ställen | (L07) |

### Kontroll 2: loss vid start
En otränad modell bör vara ungefär lika osäker på alla 10 klasser. Då är loss $-\ln(1/10) = \ln 10 \approx 2{,}30$. Ett mycket högre värde tyder på fel skala på indata eller på initieringen.

In [ ]:
set_seed(42)
fresh = make_mlp().to(device)
X, y = next(iter(train_loader))
with torch.no_grad():
    start_loss = loss_fn(fresh(X.to(device)), y.to(device)).item()
print(f"loss vid start: {start_loss:.3f}   förväntat ln(10) = {math.log(10):.3f}")

### Kontroll 3: överanpassa en batch
En modell med tillräcklig kapacitet ska kunna lära sig **en enda batch** utantill. Om den inte klarar det finns en bugg: fel etiketter, en saknad `zero_grad`, fel loss eller en frusen modell. Testet är snabbt och fångar många fel.

In [ ]:
def overfit_one_batch(model, X, y, steps=200, lr=1e-3):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    X, y = X.to(device), y.to(device)
    for step in range(steps):
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        opt.step()
    return loss.item()

set_seed(42)
X_small, y_small = X[:32], y[:32]
print(f"loss efter 200 steg på en batch med 32: {overfit_one_batch(make_mlp().to(device), X_small, y_small):.4f}")

### Kontroll 4: gradientnormer
Gradientens storlek visar om information flödar bakåt genom nätet. Exakt 0 betyder att nätet är dött. Mycket stora värden betyder exploderande gradienter, och `nan` betyder att något har gått sönder.

In [ ]:
def grad_norms(model):
    return {name: p.grad.norm().item() for name, p in model.named_parameters() if p.grad is not None}

set_seed(42)
m = make_mlp().to(device)
loss_fn(m(X.to(device)), y.to(device)).backward()
for name, g in grad_norms(m).items():
    print(f"{name:10s} gradientnorm {g:.4f}")

### Vanliga tysta buggar
**Gissa först:** vad händer om man lägger en softmax i sista lagret och sedan använder `CrossEntropyLoss` (som redan gör softmax)?

In [ ]:
set_seed(42)
with_softmax = nn.Sequential(make_mlp(), nn.Softmax(dim=1)).to(device)   # BUGG
print(f"MED softmax-bugg:  loss efter 200 steg = {overfit_one_batch(with_softmax, X_small, y_small):.4f}")
set_seed(42)
print(f"utan bugg:         loss efter 200 steg = {overfit_one_batch(make_mlp().to(device), X_small, y_small):.4f}")

Med dubbel softmax kan loss aldrig bli lägre än cirka 1,46, eftersom sannolikheter i [0, 1] är för "platta" som logits. Gradienterna blir också små. Modellen *tränar*, men sämre, och inget felmeddelande visas. Det är exakt sådana buggar som kontroll 3 fångar.

Andra vanliga buggar:
- **Glömd `optimizer.zero_grad()`**: gradienterna ackumuleras (L02), och stegen blir allt större och felriktade.
- **Etiketter som `float` i stället för `long`** till `CrossEntropyLoss`: ger fel eller tolkas som sannolikheter.
- **Data och etiketter i otakt**, till exempel efter en egen shuffle av bara `X`: accuracy fastnar runt slumpnivå.
- **Glömd normalisering**, eller olika normalisering för train och test.

## 5. Kort om initiering

Startvärdena för vikterna avgör om signalen och gradienterna får rimlig storlek genom nätet. **Vad händer om alla vikter startar på 0?**

In [ ]:
set_seed(42)
zero_model = make_mlp().to(device)
with torch.no_grad():
    for p in zero_model.parameters():
        p.zero_()                      # alla vikter och bias = 0
print(f"nollinitierad: loss efter 200 steg = {overfit_one_batch(zero_model, X_small, y_small):.4f}")
print("vikter i första lagret, unika värden:", zero_model[1].weight.unique().numel())

Med bara nollor blir ReLU-aktiveringarna noll. Inga gradienter når de tidigare lagren, och alla neuroner i ett lager förblir identiska (*symmetri*). Nätet kan inte lära sig något utöver det sista lagrets bias.

**Slumpmässig initiering** bryter symmetrin. **Skalan** på slumpen väljs så att variansen bevaras genom lagren. Det heter Kaiming eller He (för ReLU) och Xavier eller Glorot (för tanh). PyTorchs standardinitiering av `nn.Linear` och `nn.Conv2d` är bra för vanliga nät, så **du behöver sällan ändra den själv**. Detaljerna finns under "Senare fördjupning".

## 6. Övningar

### Övning 1 – spara, ladda och jämför
Träna en modell i 3 epoker. Spara dess `state_dict` till `CKPT_DIR / "ovning1.pt"`, ladda den i en ny modell och visa att val accuracy blir exakt densamma.

In [ ]:
# TODO
acc_original = None
acc_loaded = None

In [ ]:
assert acc_original == acc_loaded and acc_original > 0.8
print("Rätt!", acc_original)

### Övning 2 – buggjakt
Träningskoden nedan innehåller **tre buggar**. Den kör utan felmeddelande men lär sig dåligt. Hitta och rätta dem så att överanpassningstestet går ned under 0,05 i loss.

In [ ]:
set_seed(42)
buggy_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256), nn.ReLU(),
    nn.Linear(256, 10), nn.ReLU(),          # ← ?
)
buggy_model.to(device)
opt = torch.optim.Adam(buggy_model.parameters(), lr=1e-3)
Xb, yb = X_small.to(device), y_small.to(device)
yb_shuffled = yb[torch.randperm(len(yb), device=device)]   # ← ?

for step in range(200):
    loss = loss_fn(buggy_model(Xb), yb_shuffled)
    loss.backward()
    opt.step()                                              # ← ?
final_loss = loss.item()
print(f"slut-loss: {final_loss:.4f}")

In [ ]:
assert final_loss < 0.05, final_loss
print("Rätt! Alla buggar hittade.")

### Övning 3 – återuppta från en checkpoint i `dlkit`
Kör `fit` i 4 epoker med `checkpoint_path=CKPT_DIR / "ovning3.pt"`. Ladda sedan checkpointen och kontrollera att epoken och val loss som sparades motsvarar den **bästa** epoken i `history`.

In [ ]:
# TODO
ckpt = None
history = None

In [ ]:
best_idx = int(np.argmin(history["val_loss"]))
assert ckpt["epoch"] == best_idx + 1
assert abs(ckpt["val_loss"] - history["val_loss"][best_idx]) < 1e-9
print("Rätt! bästa epoken:", ckpt["epoch"])

## När använder jag detta?
- **`state_dict` och `torch.save`:** när du vill spara en färdig modell (steg 10) eller den bästa modellen under träningen.
- **Full checkpoint** (modell, optimerare och epok): vid långa träningar som kan avbrytas, och när du vill träna vidare.
- **TensorBoard eller en annan logg:** för *alla* experiment du vill kunna jämföra. Börja logga direkt, inte i efterhand.
- **Felsökningschecklistan:** varje gång du sätter upp en ny modell eller datapipeline, *innan* du kör en lång träning.
- **Initiering:** standardvärdena fungerar nästan alltid. Misstänk initieringen om loss vid start är långt från $\ln K$.

## Sammanfattning
- Spara `state_dict`, inte hela modellen. Ladda med samma arkitektur och anropa `.eval()`.
- En checkpoint innehåller modell, optimerare och epok.
- Logga varje körning med namn och konfiguration.
- Loss vid start ska vara ≈ $\ln K$, och en batch ska gå att överanpassa. Det fångar de flesta buggar.
- `dlkit.fit` har nu `checkpoint_path=` och `writer=`. Det är den slutliga versionen i kursen.

**Del B är klar.** Du kan nu hela arbetsflödet. I Del C byter vi ut MLP:n mot nätverk som passar bilder, sekvenser och text.